# 2️⃣ BASELINE MODELLERİ EĞİTME

Bu notebook'ta:
- 13 farklı Transfer Learning modelini eğitiyoruz
- Her modelin baseline performansını kaydediyoruz
- Modelleri kayıt ediyoruz (pruning için)
- Detaylı karşılaştırma raporu oluşturuyoruz

⏱️ **Tahmini süre**: GPU ile ~2-3 saat

## 1️⃣ KÜTÜPHANELERI YÜKLEYİN

In [ ]:
# Gerekli paketleri yükle
import sys
sys.path.append('/kaggle/working')

# Standart paketler
!pip install -r Cervical-Canser/requirements.txt -q
print("✅ Standart paketler yüklendi")

In [ ]:
# Pruning paketi (Kaggle'da özel kurulum gerekebilir)
!pip install --no-build-isolation tensorflow-model-optimization -q
print("✅ Pruning paketi yüklendi")

In [ ]:
# Modülleri import et
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import json
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Özel modüller
sys.path.insert(0, '/kaggle/working/Cervical-Canser')
from config import (
    DATA_PATH, IMAGE_SIZE, BATCH_SIZE, NUM_CLASSES, CLASS_LABELS,
    MODELS_TO_TRAIN, EPOCHS_BASELINE, LEARNING_RATE, RANDOM_SEED,
    RESULTS_DIR, MODELS_DIR, REPORTS_DIR, PLOTS_DIR
)
from utils.data_utils import load_sipakmed_dataset
from utils.model_utils import get_pretrained_model, compile_model, train_model, evaluate_model, get_model_size
from utils.visualization_utils import plot_training_history, compare_models_performance

print("✅ Tüm modüller başarıyla import edildi!")
print(f"GPU Kullanılabilir: {tf.config.list_physical_devices('GPU')}")

## 2️⃣ VERİ SETINI YÜKLEYIN

**ÖNEMLİ**: Kaggle Data sekmesinden **SipakMed** veri setini eklemeniz gerekiyor!

In [ ]:
# Veri seti yolunu kontrol et
print(f"📁 Veri seti yolu: {DATA_PATH}")
print(f"📂 İçindekiler: {os.listdir(DATA_PATH) if os.path.exists(DATA_PATH) else 'BULUNAMADI!'}")

# Alternatif yolu dene
if not os.path.exists(DATA_PATH):
    print("\n⚠️  Veri seti Kaggle default yolunda bulunamadı.")
    print("Kaggle Data sekmesinden veri setini ekleyin:")
    print("1. Notebook'un sağ tarafında 'Data' sekmesini aç")
    print("2. 'Add data' butonuna tıkla")
    print("3. 'SipakMed' veri setini ara ve ekle")
    print("4. Bu cell'i tekrar çalıştır")

In [ ]:
# Veri setini yükle
print("🚀 Veri seti yükleniyor...\n")
X_train, y_train, X_val, y_val, X_test, y_test = load_sipakmed_dataset(
    DATA_PATH, 
    image_size=IMAGE_SIZE,
    validation_split=0.2,
    test_split=0.2
)

print(f"\n✅ Veri seti başarıyla yüklendi!")
print(f"📊 Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 3️⃣ MODELLERI TÜM OLARAK EĞİTİN

Bu bölüm tüm 13 modeli eğitir. Her model için:
- ✅ Model oluşturma
- ✅ Eğitim
- ✅ Değerlendirme
- ✅ Kaydetme

In [ ]:
# Sonuçları depolamak için sözlük
baseline_results = {}
training_histories = {}
trained_models = {}

print("="*70)
print("🎯 13 TRANSFER LEARNING MODELİ EĞİTİME BAŞLANIYOU")
print("="*70)
print(f"Total Models: {len(MODELS_TO_TRAIN)}")
print(f"Classes: {NUM_CLASSES}")
print(f"Train Samples: {len(X_train)}, Val Samples: {len(X_val)}, Test Samples: {len(X_test)}")
print("="*70 + "\n")

# Her model için eğitimi başlat
for idx, model_name in enumerate(MODELS_TO_TRAIN, 1):
    print(f"\n{'='*70}")
    print(f"[{idx}/{len(MODELS_TO_TRAIN)}] MODEL: {model_name.upper()}")
    print(f"{'='*70}")
    
    try:
        # 1. Model oluştur
        print(f"\n🔨 Model oluşturuluyor...")
        model = get_pretrained_model(model_name, num_classes=NUM_CLASSES)
        
        # 2. Model derle
        print(f"⚙️  Model derleniyor...")
        model = compile_model(model, learning_rate=LEARNING_RATE)
        
        # Model parametreleri
        total_params = model.count_params()
        model_size_mb = get_model_size(model)
        
        print(f"\n📊 Model Bilgileri:")
        print(f"   - Total Parameters: {total_params:,}")
        print(f"   - Model Size: {model_size_mb:.2f} MB")
        
        # 3. Modeli eğit
        print(f"\n🚀 Model eğitiliyor ({EPOCHS_BASELINE} epoch)...")
        history = train_model(
            model, X_train, y_train, X_val, y_val,
            epochs=EPOCHS_BASELINE,
            batch_size=BATCH_SIZE,
            verbose=0
        )
        
        # 4. Modeli değerlendir
        print(f"\n📈 Model test verisi üzerinde değerlendiriliyor...")
        metrics = evaluate_model(model, X_test, y_test, batch_size=BATCH_SIZE)
        
        # Sonuçları kaydet
        baseline_results[model_name] = {
            'accuracy': float(metrics['accuracy']),
            'loss': float(metrics['loss']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'auc': float(metrics['auc']),
            'total_params': int(total_params),
            'model_size_mb': float(model_size_mb),
            'training_epochs': len(history.history['loss'])
        }
        
        training_histories[model_name] = history
        trained_models[model_name] = model
        
        # Sonuçları yazdır
        print(f"\n✅ SONUÇLAR:")
        print(f"   - Accuracy: {metrics['accuracy']:.4f}")
        print(f"   - Precision: {metrics['precision']:.4f}")
        print(f"   - Recall: {metrics['recall']:.4f}")
        print(f"   - AUC: {metrics['auc']:.4f}")
        print(f"   - Loss: {metrics['loss']:.4f}")
        
        # Modeli kaydet
        model_save_path = os.path.join(MODELS_DIR, f"{model_name}_baseline.h5")
        model.save(model_save_path)
        print(f"\n💾 Model kaydedildi: {model_save_path}")
        
    except Exception as e:
        print(f"\n❌ HATA: {model_name} modeli eğitilirken hata oluştu!")
        print(f"   Detay: {str(e)}")
        continue

print(f"\n\n{'='*70}")
print(f"✅ TÜM MODELLER EĞİTİLDİ!")
print(f"{'='*70}")
print(f"\n📊 Eğitilen Model Sayısı: {len(baseline_results)}/{len(MODELS_TO_TRAIN)}")

## 4️⃣ BASELINE PERFORMANS RAPORU

In [ ]:
# Sonuçları DataFrame'e dönüştür
results_df = pd.DataFrame(baseline_results).T
results_df = results_df.sort_values('accuracy', ascending=False)

print("\n" + "="*100)
print("📊 BASELINE MODELLERI PERFORMANS KARŞILAŞTIRMASI")
print("="*100)
print(results_df.to_string())
print("="*100)

# İstatistikler
print(f"\n📈 İSTATİSTİKLER:")
print(f"\nAccuracy:")
print(f"  - En Yüksek: {results_df['accuracy'].max():.4f} ({results_df['accuracy'].idxmax()})")
print(f"  - En Düşük: {results_df['accuracy'].min():.4f} ({results_df['accuracy'].idxmin()})")
print(f"  - Ortalama: {results_df['accuracy'].mean():.4f}")
print(f"  - Std Dev: {results_df['accuracy'].std():.4f}")

print(f"\nModel Boyutu (MB):")
print(f"  - En Büyük: {results_df['model_size_mb'].max():.2f} ({results_df['model_size_mb'].idxmax()})")
print(f"  - En Küçük: {results_df['model_size_mb'].min():.2f} ({results_df['model_size_mb'].idxmin()})")
print(f"  - Ortalama: {results_df['model_size_mb'].mean():.2f}")

print(f"\nParametre Sayısı:")
print(f"  - En Çok: {results_df['total_params'].max():,} ({results_df['total_params'].idxmax()})")
print(f"  - En Az: {results_df['total_params'].min():,} ({results_df['total_params'].idxmin()})")
print(f"  - Ortalama: {results_df['total_params'].mean():,.0f}")

In [ ]:
# CSV olarak kaydet
csv_path = os.path.join(REPORTS_DIR, 'baseline_results.csv')
results_df.to_csv(csv_path)
print(f"✅ Sonuçlar kaydedildi: {csv_path}")

# JSON olarak kaydet
json_path = os.path.join(REPORTS_DIR, 'baseline_results.json')
with open(json_path, 'w') as f:
    json.dump(baseline_results, f, indent=4)
print(f"✅ JSON kaydedildi: {json_path}")

## 5️⃣ VİZÜALİZASYON

In [ ]:
# Accuracy karşılaştırması
fig, ax = plt.subplots(figsize=(14, 6))
models_sorted = results_df.index
accuracies = results_df['accuracy'].values

colors = plt.cm.RdYlGn(np.linspace(0, 1, len(models_sorted)))
bars = ax.bar(range(len(models_sorted)), accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Baseline Modelleri - Accuracy Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_xticks(range(len(models_sorted)))
ax.set_xticklabels(models_sorted, rotation=45, ha='right')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Değerleri bar üzerine yaz
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    ax.text(i, acc + 0.02, f'{acc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'baseline_accuracy_comparison.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Grafik kaydedildi: {plot_path}")

In [ ]:
# Model Boyutu vs Accuracy
fig, ax = plt.subplots(figsize=(12, 7))

scatter = ax.scatter(results_df['model_size_mb'], results_df['accuracy'], 
                     s=200, alpha=0.7, c=results_df['accuracy'], 
                     cmap='RdYlGn', edgecolors='black', linewidth=1.5)

# Model adlarını ekle
for idx, row in results_df.iterrows():
    ax.annotate(idx, (row['model_size_mb'], row['accuracy']), 
               fontsize=9, ha='center', va='center', fontweight='bold')

ax.set_xlabel('Model Boyutu (MB)', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Model Boyutu vs Accuracy (Baseline)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Accuracy', fontweight='bold')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'model_size_vs_accuracy.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Grafik kaydedildi: {plot_path}")

In [ ]:
# Metrik Karşılaştırması (Heatmap)
metrics_to_plot = ['accuracy', 'precision', 'recall', 'auc']
metrics_data = results_df[metrics_to_plot]

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(metrics_data, annot=True, fmt='.4f', cmap='RdYlGn', 
            vmin=0, vmax=1, cbar_kws={'label': 'Score'}, ax=ax,
            linewidths=0.5, linecolor='gray')

ax.set_title('Baseline Modelleri - Detaylı Metrik Karşılaştırması', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Metrikler', fontsize=12, fontweight='bold')
ax.set_ylabel('Modeller', fontsize=12, fontweight='bold')

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'baseline_metrics_heatmap.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Heatmap kaydedildi: {plot_path}")

In [ ]:
# En iyi modelin eğitim tarihi
best_model_name = results_df['accuracy'].idxmax()
best_history = training_histories[best_model_name]

print(f"\n🏆 EN İYİ MODEL: {best_model_name.upper()}")
print(f"   Accuracy: {results_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"   Model Boyutu: {results_df.loc[best_model_name, 'model_size_mb']:.2f} MB\n")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy
axes[0, 0].plot(best_history.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(best_history.history['val_accuracy'], label='Val', linewidth=2)
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Loss
axes[0, 1].plot(best_history.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(best_history.history['val_loss'], label='Val', linewidth=2)
axes[0, 1].set_title('Loss', fontweight='bold')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Precision
axes[1, 0].plot(best_history.history['precision'], label='Train', linewidth=2)
axes[1, 0].plot(best_history.history['val_precision'], label='Val', linewidth=2)
axes[1, 0].set_title('Precision', fontweight='bold')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Recall
axes[1, 1].plot(best_history.history['recall'], label='Train', linewidth=2)
axes[1, 1].plot(best_history.history['val_recall'], label='Val', linewidth=2)
axes[1, 1].set_title('Recall', fontweight='bold')
axes[1, 1].set_ylabel('Recall')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

fig.suptitle(f'En İyi Model - {best_model_name.upper()} Eğitim Tarihi', 
            fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, f'{best_model_name}_training_history.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✅ Eğitim tarihi grafiği kaydedildi: {plot_path}")

## 6️⃣ ÖZET RAPOR

In [ ]:
# Detaylı rapor oluştur
report = f"""
╔{'='*88}╗
║{'BASELINE MODELLERI EĞİTİM RAPORU':^88}║
╚{'='*88}╝

📅 Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

{'─'*90}
📊 VERİ SETİ BİLGİLERİ
{'─'*90}
  • Total Görüntü: {len(X_train) + len(X_val) + len(X_test)}
  • Train: {len(X_train)} ({len(X_train)/(len(X_train) + len(X_val) + len(X_test))*100:.1f}%)
  • Validation: {len(X_val)} ({len(X_val)/(len(X_train) + len(X_val) + len(X_test))*100:.1f}%)
  • Test: {len(X_test)} ({len(X_test)/(len(X_train) + len(X_val) + len(X_test))*100:.1f}%)
  • Sınıf Sayısı: {NUM_CLASSES}
  • Görüntü Boyutu: {IMAGE_SIZE}x{IMAGE_SIZE}x3

{'─'*90}
🎯 MODELLER
{'─'*90}
  Toplam Model Sayısı: {len(MODELS_TO_TRAIN)}
  Eğitilen Model Sayısı: {len(baseline_results)}
  
  Modeller:
"""

for i, model in enumerate(MODELS_TO_TRAIN, 1):
    report += f"    {i:2d}. {model}\n"

report += f"""
{'─'*90}
🏆 PERFORMANCE ÖZETİ
{'─'*90}

  EN İYİ MODELLER (Accuracy'ye göre):
"""

for i, (idx, row) in enumerate(results_df.head(3).iterrows(), 1):
    report += f"    {i}. {idx:20s} - Accuracy: {row['accuracy']:.4f}, Size: {row['model_size_mb']:.2f} MB\n"

report += f"""
  EN KÜÇÜK MODELLER (Boyuta göre):
"""

for i, (idx, row) in enumerate(results_df.nsmallest(3, 'model_size_mb').iterrows(), 1):
    report += f"    {i}. {idx:20s} - Size: {row['model_size_mb']:6.2f} MB, Accuracy: {row['accuracy']:.4f}\n"

report += f"""
{'─'*90}
📈 İSTATİSTİKLER
{'─'*90}

  Accuracy:
    • Ortalama: {results_df['accuracy'].mean():.4f}
    • En Yüksek: {results_df['accuracy'].max():.4f}
    • En Düşük: {results_df['accuracy'].min():.4f}
    • Std Dev: {results_df['accuracy'].std():.4f}

  Model Boyutu (MB):
    • Ortalama: {results_df['model_size_mb'].mean():.2f}
    • En Büyük: {results_df['model_size_mb'].max():.2f}
    • En Küçük: {results_df['model_size_mb'].min():.2f}

  Toplam Parametreler:
    • Ortalama: {results_df['total_params'].mean():,.0f}
    • En Çok: {results_df['total_params'].max():,}
    • En Az: {results_df['total_params'].min():,}

{'─'*90}
💾 ÇIKTI DOSYALARI
{'─'*90}
  • CSV Raporu: {csv_path}
  • JSON Raporu: {json_path}
  • Grafikler: {PLOTS_DIR}
  • Modeller: {MODELS_DIR}

{'─'*90}
✅ BAŞARILI
{'─'*90}
"""

print(report)

# Raporu dosyaya kaydet
report_path = os.path.join(REPORTS_DIR, 'baseline_training_report.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"✅ Rapor kaydedildi: {report_path}")

In [ ]:
# Pickle ile modelleri ve history'leri kaydet (pruning için)
pickle_path = os.path.join(MODELS_DIR, 'trained_models_and_histories.pkl')
with open(pickle_path, 'wb') as f:
    pickle.dump({
        'models': trained_models,
        'histories': training_histories,
        'results': baseline_results
    }, f)

print(f"✅ Modeller ve history'ler kaydedildi: {pickle_path}")
print(f"\n✅ TÜKTÜN BAŞARILI! Sonraki adım: 03_pruning_optimization.ipynb")